# NER Model Training with spaCy

This notebook guides you through the process of:
1. Setting up a spaCy model for NER training.
2. Loading annotated data.
3. Training the NER model.
4. Evaluating the model's performance.

In [11]:
# Install necessary packages
!pip install spacy
!python -m spacy download en_core_web_sm


[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: C:\Users\ASUS\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: C:\Users\ASUS\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     --------------------------------------- 0.0/12.8 MB 165.2 kB/s eta 0:01:18
     --------------------------------------- 0.0/12.8 MB 163.8 kB/s eta 0:01:18
     --------------------------------------- 0.1/12.8 MB 219.0 kB/s eta 0:00:59
      -------------------------------------- 0.2/12.8 MB 811.5 kB/s eta 0:00:16
     - -------------------------------------- 0.6/12.8 MB 1.9 MB/s eta 0:00:07
     -------- ------------------------------- 2.6/12.8 MB 7.1 MB/s eta 0:00:02
     ----------- ---------------------------- 3.7/12.8 MB 9.1 MB/s eta 0:00:02
     ------------------ --------------------- 5.8/12.8 MB 12.7 MB/s eta 0:00:01
     -------------------- ------------------- 6.7/12.8 MB 13.4 MB/s eta 0:00:01
     ----------------------------- ---------- 9.4/12.8 MB 17.1 MB/s eta 0:00:01
     --------------------------------- ----- 11.0/12.8 MB 

## Step 1: Set Up spaCy for NER Training

We'll start by initializing a blank English model and adding an NER pipeline.

In [2]:
import spacy

# Create a blank English model
nlp = spacy.blank("en")

# Add the Named Entity Recognition (NER) pipeline
ner = nlp.add_pipe("ner")

# Add labels to the NER pipeline based on your data
ner.add_label("TASK")
ner.add_label("SKILLS")

print("spaCy model and NER pipeline set up successfully.")

spaCy model and NER pipeline set up successfully.


## Step 2: Load the Annotated Data

Next, we'll load the JSON files that contain the annotated data from both datasets (profile roles and course descriptions).

In [4]:
import json

# Load the annotated skill roles data
with open('..\\annotated_skill_roles_spacy.json', 'r') as f:
    train_data_roles = json.load(f)

# Load the annotated course descriptions data
with open('..\\annotated_course_details_spacy.json', 'r') as f:
    train_data_courses = json.load(f)

# Combine the training data from both datasets
train_data = train_data_roles + train_data_courses

print(f"Loaded {len(train_data_roles)} examples from skill roles dataset.")
print(f"Loaded {len(train_data_courses)} examples from course descriptions dataset.")
print(f"Total training examples: {len(train_data)}")

Loaded 36 examples from skill roles dataset.
Loaded 736 examples from course descriptions dataset.
Total training examples: 772


In [6]:
# Step 3: Remove Overlapping Entities

# Function to remove overlapping entities
def remove_overlapping_entities(annotations):
    # Sort entities by their start positions
    sorted_entities = sorted(annotations['entities'], key=lambda x: x[0])
    
    # List to hold non-overlapping entities
    filtered_entities = []
    last_end = -1

    # Iterate through sorted entities
    for start, end, label in sorted_entities:
        if start >= last_end:  # No overlap with the previous entity
            filtered_entities.append((start, end, label))
            last_end = end
        else:
            # Overlapping entity detected; handle it based on strategy
            # Here, we'll skip the overlapping entity, but you could adjust it if needed
            continue

    return filtered_entities

# Clean the data by removing overlaps
for i, (text, annotations) in enumerate(train_data):
    annotations['entities'] = remove_overlapping_entities(annotations)

print("Overlapping entities removed.")


Overlapping entities removed.


## Step 3: Train the NER Model

We will now train the model using the annotated data. This involves multiple iterations to update the model’s weights and improve its ability to recognize `TASK` and `SKILLS` entities.

In [7]:
# Step 4: Train the NER Model

from spacy.training.example import Example

# Start the training process
optimizer = nlp.begin_training()
n_iter = 10  # Number of training iterations

for itn in range(n_iter):
    print(f"Starting iteration {itn+1}/{n_iter}")
    losses = {}
    for text, annotations in train_data:
        doc = nlp.make_doc(text)
        example = Example.from_dict(doc, annotations)
        nlp.update([example], sgd=optimizer, losses=losses)
    print(f"Iteration {itn+1} Losses: {losses}")

# Save the trained model to disk
nlp.to_disk("ner_model_task_skills_cleaned")

print("Training complete. Model saved as 'ner_model_task_skills_cleaned'.")


Starting iteration 1/10


C:\Users\ASUS\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text " Define, implement, communicate and maintain cyber..." with entities "[(1, 7, 'TASK'), (9, 18, 'TASK'), (20, 31, 'TASK')...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
C:\Users\ASUS\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Oversees and assures compliance with cybersecurity..." with entities "[(37, 50, 'SKILLS'), (76, 97, 'SKILLS'), (210, 225...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. 

Iteration 1 Losses: {'ner': 6246.866258099457}
Starting iteration 2/10
Iteration 2 Losses: {'ner': 4473.027402974749}
Starting iteration 3/10
Iteration 3 Losses: {'ner': 4102.592972145425}
Starting iteration 4/10
Iteration 4 Losses: {'ner': 3684.8540313429075}
Starting iteration 5/10
Iteration 5 Losses: {'ner': 3346.3309348721664}
Starting iteration 6/10
Iteration 6 Losses: {'ner': 2935.621605054106}
Starting iteration 7/10
Iteration 7 Losses: {'ner': 2510.9777881721766}
Starting iteration 8/10
Iteration 8 Losses: {'ner': 2341.0707236200383}
Starting iteration 9/10
Iteration 9 Losses: {'ner': 2113.6501647832556}
Starting iteration 10/10
Iteration 10 Losses: {'ner': 1844.9456083150314}
Training complete. Model saved as 'ner_model_task_skills_cleaned'.


## Next Steps

1. **Evaluate the Model**: After training, we can evaluate the model to check its performance on identifying entities.
2. **Use the Model**: Apply the model to new text data to extract `TASK` and `SKILLS` entities.

In [8]:
# Load the trained model
nlp_ner = spacy.load("ner_model_task_skills_cleaned")

# Sample text for evaluation (replace with your own text)
sample_text = "This course teaches students how to conduct a penetration test using the latest tools."

# Process the sample text with the trained model
doc = nlp_ner(sample_text)

# Display the entities recognized by the model
for ent in doc.ents:
    print(ent.text, ent.label_)


conduct TASK


In [10]:
# Load the trained model
nlp_ner = spacy.load("ner_model_task_skills_cleaned")

# Function to extract entities from text
def extract_entities(text):
    doc = nlp_ner(text)
    return [(ent.text, ent.label_) for ent in doc.ents]

# Example: Extract entities from a new course description
new_course_description = "This course covers network security principles and teaches students how to secure a network using firewalls."
entities = extract_entities(new_course_description)

# Display the extracted entities
for entity in entities:
    print(entity)


('secure', 'TASK')
